# $\mathbb{Z}_2^F \times \mathbb{Z}_2^T$ Hamiltonians

Created: 14-06-2026

Build off [this notebook](../fermionic_cases/time_reversal_symmetries/z2_f_x_z2_t_hamiltonians.ipynb), now sweep and save states.

# Imports

In [1]:
from time import time

In [2]:
import numpy as np

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from itertools import product

In [6]:
import quimb.tensor as qtn
import quimb as qu

In [7]:
from quspin.operators import hamiltonian
from quspin.operators import quantum_operator
from quspin.basis import spin_basis_1d, spinless_fermion_basis_1d, tensor_basis

In [8]:
from humanize import naturalsize

# Definitions

In [9]:
group_quads = list(product([0,1], repeat=4))

In [10]:
spin_ops_dict = {
    (0,0): [("I", 1), ("y", 1)],
    (1,1): [("I", 1), ("y", -1)],
    (0,1): [("z", 1), ("x", 1j)],
    (1,0): [("z", 1), ("x", -1j)]
}

In [11]:
def get_fermionic_op_string(group_quad):
    g_left, g_in, g_out, g_right = group_quad

    out_string = ''
    out_indices = list()

    if (g_left + g_out) % 2:
        out_string += '+'
        out_indices.append(0)
    if (g_out + g_right) % 2:
        out_string += '+'
        out_indices.append(1)
    if (g_in + g_right) % 2:
        out_string += '-'
        out_indices.append(1)
    if (g_left + g_in) % 2:
        out_string += '-'
        out_indices.append(0)

    return (out_string, out_indices)

In [12]:
def cocycle_phase(group_quad):
    g_left, g_in, g_out, g_right = group_quad

    exponent = (
        (g_in - g_left)*(g_right - g_in)
        - (g_out - g_left)*(g_right - g_out)
    )
    exponent = exponent % 2

    # i.e. (-1)**exponent
    out = -1 if exponent else 1

    return out

In [21]:
def get_group_quad_terms(group_quad, L, nontriv_cocycle=False,
                         strength_scaling=1):
    g_left, g_in, g_out, g_right = group_quad

    left_op = spin_ops_dict[(g_left, g_left)]
    mid_op = spin_ops_dict[(g_out, g_in)]
    right_op = spin_ops_dict[(g_right, g_right)]

    op_triples = product(left_op, mid_op, right_op)

    terms = list()

    for left_pair, mid_pair, right_pair in op_triples:
        left_string, left_strength = left_pair
        mid_string, mid_strength = mid_pair
        right_string, right_strength = right_pair

        spin_string = f"{left_string}{mid_string}{right_string}"
        strength = -(1/16)*left_strength*mid_strength*right_strength

        ferm_op_string, ferm_indices = get_fermionic_op_string(group_quad)
        op_string = f"{spin_string}|{ferm_op_string}"

        base_index = [0, 1, 2, *ferm_indices]
        all_indices = [
            [(x+i)%L for x in base_index]
            for i in range(L)
        ]

        phase = cocycle_phase(group_quad) if nontriv_cocycle else 1
        scaling_constant = strength*strength_scaling*phase
        current_term = [
            op_string, [[scaling_constant, *indices] for indices in all_indices]
        ]

        terms.append(current_term)

    return terms

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [14]:
triv_fermion_terms_phases = [
    ('I', 1),
    ('n', -1)
]

In [15]:
def get_trivial_group_quad_terms(group_quad, L, strength_scaling=1):
    #Case where both cocycle and fermionic decoration are trivial.

    g_left, g_in, g_out, g_right = group_quad

    left_op = spin_ops_dict[(g_left, g_left)]
    mid_op = spin_ops_dict[(g_out, g_in)]
    right_op = spin_ops_dict[(g_right, g_right)]

    op_triples = product(left_op, mid_op, right_op)

    terms = list()

    for left_pair, mid_pair, right_pair in op_triples:
        left_string, left_strength = left_pair
        mid_string, mid_strength = mid_pair
        right_string, right_strength = right_pair

        spin_string = f"{left_string}{mid_string}{right_string}"
        strength = -(1/16)*left_strength*mid_strength*right_strength


        for pair_1, pair_2 in product(triv_fermion_terms_phases, repeat=2):
            fp_op_1, fp_phase_1 = pair_1
            fp_op_2, fp_phase_2 = pair_2
            
            op_string = f"{spin_string}|{fp_op_1}{fp_op_2}"
    
            base_index = [0, 1, 2, 0, 1]
            all_indices = [
                [(x+i)%L for x in base_index]
                for i in range(L)
            ]
    
            scaling_constant = strength*strength_scaling*fp_phase_1*fp_phase_2
            current_term = [
                op_string, [[scaling_constant, *indices] for indices in all_indices]
            ]
    
            terms.append(current_term)

    return terms

In [16]:
def get_triv_to_n1_non_triv_hamiltonian(t, L):
    spin_basis = spin_basis_1d(L)
    fermion_basis = spinless_fermion_basis_1d(L)
    basis = tensor_basis(spin_basis, fermion_basis)

    triv_terms = [
        l for group_quad in group_quads
        for l in get_trivial_group_quad_terms(group_quad, L, 1-t)
    ]

    non_triv_terms = [
        l for group_quad in group_quads
        for l in get_group_quad_terms(group_quad, L, False, t)
    ]

    all_terms = triv_terms + non_triv_terms

    h = hamiltonian(
        all_terms,
        [],
        basis=basis,
        dtype=np.complex128,
        check_symm=False,
        check_herm=False
    )

    return h

In [17]:
def get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L):
    spin_basis = spin_basis_1d(L)
    fermion_basis = spinless_fermion_basis_1d(L)
    basis = tensor_basis(spin_basis, fermion_basis)

    triv_terms = [
        l for group_quad in group_quads
        for l in get_group_quad_terms(group_quad, L, False, 1-t)
    ]

    non_triv_terms = [
        l for group_quad in group_quads
        for l in get_group_quad_terms(group_quad, L, True, t)
    ]

    all_terms = triv_terms + non_triv_terms

    h = hamiltonian(
        all_terms,
        [],
        basis=basis,
        dtype=np.complex128,
        check_symm=False,
        check_herm=False
    )

    return h

# Sweep

In [18]:
parameters = np.linspace(0, 1, 21)

## 8 site

In [19]:
L = 8

## Trivial cocycle

In [22]:
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_triv_to_nontriv_n1_8_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                                         | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_17896/2657243103.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
  5%|█████████▏                                                                                                                                                                                       | 1/21 [00:00<00:06,  2.87it/s]/tmp/ipykernel_17896/2657243103.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|███████████████████████████████████████████████████████████████████████████████████

## Nontrivial cocycle

In [20]:
for t in tqdm(parameters):
    h = get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_nontriv_n1_to_nontriv_cocyle_8_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                                         | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_5301/1466521248.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:43<00:00,  2.08s/it]


## 10 site

In [23]:
L = 10

## Trivial cocycle

In [23]:
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_triv_to_nontriv_n1_10_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                              | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_17896/2657243103.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:45<00:00,  2.17s/it]


## Nontrivial cocycle

In [25]:
for t in tqdm(parameters):
    h = get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_nontriv_n1_to_nontriv_cocyle_10_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                                         | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_5301/1466521248.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [15:34<00:00, 44.49s/it]


## 12 site
This takes a while...

In [26]:
L = 12

## Trivial cocycle

In [19]:
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_triv_to_nontriv_n1_12_site_ed/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                                                                                                                         | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_3403/2504458043.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
  5%|█████████▏                                                                                                                                                                                       | 1/21 [00:55<18:20, 55.03s/it]/tmp/ipykernel_3403/2504458043.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
  5%|████████▋                                                                            

KeyboardInterrupt: 

## Nontrivial cocycle

In [ ]:
for t in tqdm(parameters):
    h = get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_nontriv_n1_to_nontriv_cocyle_12_site_ed/{t_string}.npz', energy=e, psi=psi)